# Fase 3 · Pipeline y modelado con PySpark

**Proyecto integrador de tecnologías de Big Data** · Predicción de delitos en Chicago · Valentino De Paola, Eduardo López y Carlos David González

> **Pregunta de la Fase 2:** ¿En qué zonas, días y horarios de Chicago es más probable que ocurra un delito en las próximas semanas, y qué tipo de delito será?

Este notebook ejecuta el plan CRISP-DM de la Fase 2 sobre el clúster (HDFS + Spark sobre YARN):

- **A. Preparación de datos** (CRISP-DM III): carga, limpieza, transformación y guardado en Parquet.
- **B. Modelado analítico** (CRISP-DM IV): dos modelos de SparkML y su comparación.
- **C. Visualización:** cinco gráficas que responden las preguntas analíticas.

**Cómo correrlo:** en la instancia, con `LOCAL = False`, *Kernel → Restart Kernel and Run All Cells*. El CSV crudo lo deja en HDFS `scripts/ingesta-raw.sh`, que imprime la fecha que va en `FECHA_RAW`.

## Configuración

In [ ]:
LOCAL = False                 # False: instancia (HDFS + YARN) · True: prueba en una compu con el CSV local
FECHA_RAW = "2026-10-23"      # carpeta de la ingesta en HDFS (la imprime scripts/ingesta-raw.sh)

RUTA_RAW_LOCAL = "../data/Crimes_2025.csv"
RUTA_RAW_HDFS = f"hdfs:///datalake/raw/crimes_2025/fecha={FECHA_RAW}/Crimes_2025.csv"
RUTA_RAW = RUTA_RAW_LOCAL if LOCAL else RUTA_RAW_HDFS

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

builder = (SparkSession.builder
           .appName(f"fase3-pipeline-{'local' if LOCAL else 'cluster'}")
           .config("spark.sql.session.timeZone", "UTC"))

if LOCAL:
    spark = (builder.master("local[*]")
             .config("spark.driver.memory", "2g")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
else:
    spark = builder.enableHiveSupport().getOrCreate()   # master = yarn (spark-defaults.conf)

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", spark.sparkContext.master)

# A. Preparación de datos con PySpark · CRISP-DM Fase III

## A1. Carga e inspección inicial

### Paso 1 · Carga desde HDFS

Se lee el CSV crudo tal como lo entrega el portal de datos de Chicago, con `inferSchema=True` para ver qué tipos detecta Spark por su cuenta. Los tipos definitivos (por ejemplo, convertir `Date` de texto a fecha) se fijan en A2.

In [ ]:
df_raw = spark.read.csv(RUTA_RAW, header=True, inferSchema=True)
print("Leído desde:", RUTA_RAW)

### Paso 2 · Exploración rápida

Esquema, tamaño y estadísticas descriptivas.

In [ ]:
df_raw.printSchema()

filas = df_raw.count()
print(f"Filas: {filas:,} · Columnas: {len(df_raw.columns)}")

Estadísticas de las columnas numéricas. Ojo: de ellas, solo las coordenadas son medidas de verdad; `ID`, `Beat`, `District`, `Ward`, `Community Area` y `Year` son códigos que Spark lee como números, así que su media o desviación no tienen sentido.

In [ ]:
numericas = [c.name for c in df_raw.schema.fields if isinstance(c.dataType, NumericType)]
estadisticas = (df_raw.select(numericas).describe().toPandas()
                .set_index("summary").T.apply(pd.to_numeric).round(2))
estadisticas

Cuántos valores distintos tiene cada variable categórica clave, y los tipos de delito más frecuentes.

In [ ]:
categoricas = ["Primary Type", "Description", "Location Description", "IUCR", "FBI Code",
               "Beat", "District", "Ward", "Community Area"]
distintos = (df_raw.agg(*[F.countDistinct(c).alias(c) for c in categoricas])
             .toPandas().T.rename(columns={0: "valores distintos"}))
distintos

In [ ]:
top_tipos = (df_raw.groupBy("Primary Type").count()
             .orderBy(F.desc("count")).limit(10)
             .withColumn("% del total", F.round(F.col("count") * 100 / filas, 2))
             .toPandas())
top_tipos

### Paso 3 · Tabla de variables

Para cada variable: tipo que detectó Spark, nulos (vacíos incluidos) en cantidad y porcentaje, y observaciones. Las observaciones combinan lo que se mide aquí con las decisiones del plan de la Fase 2.

In [ ]:
# Nulos y vacíos de todas las columnas en una sola pasada
nulos = df_raw.agg(*[
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
]).first().asDict()

# Mediciones que se citan en las observaciones
hora_exacta = df_raw.filter(F.col("Date").endswith(" 12:00:00 AM") | F.col("Date").endswith(" 12:00:00 PM")).count()
ids_repetidos = filas - df_raw.select("ID").distinct().count()
casos_repetidos = df_raw.groupBy("Case Number").count().filter("count > 1").count()
anios = sorted(r[0] for r in df_raw.select("Year").distinct().collect())
pct_arresto = df_raw.filter(F.col("Arrest")).count() * 100 / filas
pct_domestico = df_raw.filter(F.col("Domestic")).count() * 100 / filas
pct_top2 = top_tipos["% del total"].head(2).sum()
fecha = F.try_to_timestamp("Date", F.lit("MM/dd/yyyy hh:mm:ss a"))
fechas = df_raw.select(F.date_format(F.min(fecha), "dd/MM/yyyy"), F.date_format(F.max(fecha), "dd/MM/yyyy")).first()
ward_cero = df_raw.filter(F.col("Ward") == 0).count()
area_cero = df_raw.filter(F.col("Community Area") == 0).count()
distritos_raros = (df_raw.groupBy("District").count().filter("count < 1000")
                   .orderBy("District").collect())   # los 22 distritos regulares tienen miles de registros cada uno
n = distintos["valores distintos"]

observaciones = {
    "ID": f"Identificador único del incidente; {ids_repetidos} repetidos.",
    "Case Number": f"Número de caso policial; {casos_repetidos} se repiten: varios incidentes de un mismo caso, no errores de carga (Fase 2).",
    "Date": f"Fecha y hora. Spark la lee como texto (MM/dd/yyyy hh:mm:ss AM/PM); se convierte en A2. Rango: {fechas[0]} a {fechas[1]}. "
            f"El {hora_exacta * 100 / filas:.2f} % cae exactamente a medianoche o mediodía: hora anotada cuando no se conoce la real.",
    "Block": "Cuadra aproximada (anonimizada). No se usa en el modelo.",
    "IUCR": f"Código de delito de Illinois ({n['IUCR']} valores). Redundante con Primary Type y Description.",
    "Primary Type": f"Tipo de delito ({n['Primary Type']} categorías): responde el 'qué'. Desbalanceado: las dos más comunes suman {pct_top2:.1f} %.",
    "Description": f"Subtipo de delito ({n['Description']} valores). Redundante con Primary Type para modelar.",
    "Location Description": f"Tipo de lugar ({n['Location Description']} valores). Los vacíos se imputan como DESCONOCIDO en A2.",
    "Arrest": f"Hubo arresto en el {pct_arresto:.1f} % de los casos. Muy alto en delitos detectados por patrullaje (sesgo, Fase 2).",
    "Domestic": f"Violencia doméstica en el {pct_domestico:.1f} % de los casos.",
    "Beat": f"Zona de patrullaje ({n['Beat']} valores): unidad espacial del modelo. Leída como número, pierde el cero inicial (0931 → 931).",
    "District": f"Distrito policial ({n['District']} valores). Códigos poco frecuentes: "
                + ", ".join(f"{r['District']} ({r['count']} registros)" for r in distritos_raros) + "; se revisan en A2.",
    "Ward": f"Distrito electoral ({n['Ward']} valores). {ward_cero} registros con código 0, que no existe. No se usa en el modelo.",
    "Community Area": f"Área comunitaria ({n['Community Area']} valores): nivel de las gráficas por zona. {area_cero} registros con código 0 (las áreas van de 1 a 77); se revisan en A2.",
    "FBI Code": f"Clasificación del FBI ({n['FBI Code']} valores). Redundante con Primary Type.",
    "X Coordinate": "Coordenada en plano métrico (State Plane de Illinois, en pies). Se conserva para el clustering (Fase 2); los vacíos se imputan con el centro del beat en A2.",
    "Y Coordinate": "Igual que X Coordinate.",
    "Year": f"Año del incidente; valores: {anios}. Constante: se elimina en A2.",
    "Updated On": "Fecha de la última actualización en el portal. Metadato que filtraría información posterior al hecho: se elimina en A2.",
    "Latitude": "Coordenada geográfica. Mismos vacíos que X/Y.",
    "Longitude": "Coordenada geográfica. Mismos vacíos que X/Y.",
    "Location": "Punto 'POINT (lon lat)', redundante con Latitude y Longitude: se elimina en A2.",
}

tabla_variables = pd.DataFrame([
    {"variable": c, "tipo": t, "nulos": nulos[c], "% nulos": round(nulos[c] * 100 / filas, 2), "observaciones": observaciones[c]}
    for c, t in df_raw.dtypes
])
tabla_variables

### Hallazgos de A1

_Se completa con los números de la corrida en la instancia._

---

## Cierre

In [ ]:
spark.stop()   # libera la sesión: en la instancia solo cabe una sesión de Spark a la vez